# TradeFlowGNN: Model Evaluation & Comparison

This notebook compares the Graph Neural Network (GCN) results against traditional baselines:
1. **Gravity Model (OLS)**: The standard log-linear model used in economics.
2. **MLP Baseline**: A feed-forward network that ignores graph structure.

In [ ]:
# ── Colab Managed Setup (Hosted Runtime Only) ──────────────────────
import sys
from pathlib import Path
import os

def setup_colab(repo_url="https://github.com/tolyaho/TradeFlowGCN.git", branch="main"):
    if 'google.colab' in str(get_ipython()):
        if Path("/content").exists() and not Path("src").exists():
            print(f"Running in Hosted Colab. Cloning {branch} branch...")
            !pip install -q torch-geometric pytorch-lightning pyarrow pandas numpy matplotlib seaborn tqdm pyyaml scikit-learn
            repo_name = "TradeFlowGCN"
            if not Path(repo_name).exists():
                !git clone -b {branch} {repo_url}
            os.chdir(repo_name)
            
        if Path(".git").exists():
            print("Pulling latest code...")
            !git pull
        
        if os.getcwd() not in sys.path: sys.path.append(os.getcwd())
        src_path = os.path.join(os.getcwd(), "src")
        if src_path not in sys.path: sys.path.append(src_path)
    else:
        print("Running in Local Runtime. No cloning needed.")

setup_colab(branch="main")

# ── Robust Path Setup ──────────────────────────────────────────────
curr_path = Path(".").resolve()
root_path = curr_path
while not (root_path / "src").exists() and root_path.parent != root_path:
    root_path = root_path.parent

if (root_path / "src").exists():
    if str(root_path / "src") not in sys.path: sys.path.append(str(root_path / "src"))
    print(f"Project Root: {root_path}")
else:
    print(f"Warning: Could not find 'src' directory relative to {curr_path}")

## 0. Training GCN (Optional)
Run this if you don't have a checkpoint in `lightning_logs/`.

In [ ]:
!python scripts/download_data.py
!python scripts/train.py --config configs/default.yaml

## 1. Load Data & Trained GCN

In [ ]:
import torch
import numpy as np
import pandas as pd
from trade_flow_gcn.utils.config import load_config
from trade_flow_gcn.data.preprocessing import preprocess_pipeline
from trade_flow_gcn.data.dataset import build_graphs_from_dataframe, TradeDataModule
from trade_flow_gcn.models.gcn import TradeFlowGCN
from trade_flow_gcn.training.lightning_module import TradeFlowModule

# Load config and data
config = load_config(str(root_path / "configs/default.yaml"))
df = preprocess_pipeline(None, config) # Will load from cache if exists
graphs = build_graphs_from_dataframe(df, config['data']['countries'], config)

dm = TradeDataModule(
    graphs=graphs,
    train_years=tuple(config['data']['train_years']),
    val_years=tuple(config['data']['val_years']),
    test_years=tuple(config['data']['test_years'])
)
dm.setup()

# Find latest checkpoint
ckpt_files = list((root_path / "lightning_logs").glob("**/checkpoints/*.ckpt"))
if ckpt_files:
    latest_ckpt = sorted(ckpt_files, key=lambda p: p.stat().st_mtime)[-1]
    model = TradeFlowGCN(
        node_input_dim=len(config['data']['node_features']),
        edge_input_dim=len(config['data']['edge_features']),
        hidden_dim=64,
        num_gnn_layers=3
    )
    gcn_module = TradeFlowModule.load_from_checkpoint(latest_ckpt, model=model)
    gcn_module.eval()
    print(f"Loaded GCN from {latest_ckpt.name}")
else:
    print("No GCN checkpoint found. Run training first.")

## 2. Gravity Baseline (OLS)
We fit the OLS model on the training years and evaluate on the test years.

In [ ]:
from trade_flow_gcn.models.gravity_baseline import GravityBaseline

def extract_numpy(data_list):
    X_src, X_dst, E_attr, Y = [], [], [], []
    for graph in data_list:
        src, dst = graph.edge_index
        X_src.append(graph.x[src].numpy())
        X_dst.append(graph.x[dst].numpy())
        E_attr.append(graph.edge_attr.numpy())
        Y.append(graph.y.numpy())
    return (
        np.concatenate(X_src),
        np.concatenate(X_dst),
        np.concatenate(E_attr),
        np.concatenate(Y)
    )

# Split data
x_s_train, x_d_train, e_train, y_train = extract_numpy(dm.train_graphs)
x_s_test, x_d_test, e_test, y_test = extract_numpy(dm.test_graphs)

# Fit OLS
gravity = GravityBaseline()
gravity.fit(x_s_train, x_d_train, e_train, y_train)
gravity_metrics = gravity.evaluate(x_s_test, x_d_test, e_test, y_test)
print("Gravity OLS Metrics:", gravity_metrics)

## 3. Comparison Summary

In [ ]:
results = []
results.append({"Model": "Gravity (OLS)", **gravity_metrics})

if 'gcn_module' in locals():
    # Evaluate GCN on test set
    import torch_geometric
    test_loader = torch_geometric.loader.DataLoader(dm.test_graphs, batch_size=1)
    gcn_preds, gcn_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            out = gcn_module(batch)
            gcn_preds.append(out.numpy())
            gcn_targets.append(batch.y.numpy())
    
    gcn_preds = np.concatenate(gcn_preds)
    gcn_targets = np.concatenate(gcn_targets)
    
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    gcn_metrics = {
        "rmse": float(np.sqrt(mean_squared_error(gcn_targets, gcn_preds))),
        "mae": float(mean_absolute_error(gcn_targets, gcn_preds)),
        "r2": float(r2_score(gcn_targets, gcn_preds))
    }
    results.append({"Model": "TradeFlow GCN", **gcn_metrics})

summary_df = pd.DataFrame(results)
display(summary_df)